# GrowSari — Demand Forecasting (v4, validated + S3 + custom week)

**Run order: top to bottom.**

**What's new vs v3:**
1. **Dynamic validation** — every uploaded file (demand history, SKU master, combinations) is checked for
   required columns, dtypes, parseable dates, and unknown aggregation-level references *before* the engine
   runs, with clear error messages instead of cryptic pandas errors later.
2. **S3 support** — any input can be a local upload **or** an `s3://bucket/key.csv` URL (handles large files
   via chunked/streamed reads). Credentials are entered once below — they are **never** hardcoded in this
   notebook.
3. **Custom weekly calendar** — "weekly" now means the client's custom 5-day bucket (days 1-5, 6-10, 11-15,
   16-20, 21-25, 26-end-of-month → 6 buckets/month), labelled by the bucket's last day, matching the requested
   output format (`2026-03-05 ... 2026-03-31, 2026-04-05 ...`).
4. The export sheet `Output_CustomWeek` matches the client's requested shape: depot · sku · model · data_type
   (test/forecast) · date · month · year.


In [ ]:
# Colab: uncomment on first run
!pip install -q pandas numpy scikit-learn xgboost lightgbm statsmodels openpyxl matplotlib ipywidgets boto3

# growsari_engine.py must be in the same folder as this notebook (or uploaded to /content).
# If you're running this fresh in Colab, upload growsari_engine.py via the Files pane first,
# or fetch it from your own repo/storage.
import sys, os
assert os.path.exists("growsari_engine.py"), (
    "growsari_engine.py not found next to this notebook — upload it via the Colab Files pane "
    "(left sidebar) before running this cell."
)
sys.path.insert(0, ".")

import warnings, time, io, numpy as np, pandas as pd, matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
import growsari_engine as eng
from growsari_engine import ValidationError

try:
    import ipywidgets as W
    from IPython.display import display
    HAS_WIDGETS = True
except Exception:
    HAS_WIDGETS = False
print("ready · widgets:", HAS_WIDGETS)


## 1 · AWS credentials (only needed if you use s3:// URLs)

**Never hardcode keys in this notebook.** Paste credentials into the masked fields below — they live only in this runtime's memory for this session. If you've already rotated/disabled any key that was shared in chat or elsewhere, generate a fresh one in IAM and use that.

In [ ]:
AWS_ACCESS_KEY_ID = ""      # leave blank to use environment vars / IAM role instead
AWS_SECRET_ACCESS_KEY = ""  # leave blank to use environment vars / IAM role instead
AWS_REGION = "ap-south-1"

if HAS_WIDGETS:
    _ak = W.Password(description="Access Key ID:")
    _sk = W.Password(description="Secret Access Key:")
    _rg = W.Text(description="Region:", value=AWS_REGION)
    display(_ak, _sk, _rg)
    print("Fill in the boxes above, then run the NEXT cell to capture them (or just edit the variables directly).")


In [ ]:
# Run this AFTER filling the widgets above (or just set the three variables directly).
if HAS_WIDGETS:
    AWS_ACCESS_KEY_ID = _ak.value or AWS_ACCESS_KEY_ID
    AWS_SECRET_ACCESS_KEY = _sk.value or AWS_SECRET_ACCESS_KEY
    AWS_REGION = _rg.value or AWS_REGION
print("AWS region set to:", AWS_REGION, "| key configured:", bool(AWS_ACCESS_KEY_ID))


## 2 · Load & validate demand history + SKU master

Each field accepts **either** a local path (e.g. `/content/demand_history.csv`, or a Colab upload) **or** an `s3://bucket/key.csv` URL.

In [ ]:
DEMAND_SOURCE = "/content/demand_history.csv"   # or "s3://ds-stocksense-dev/.../demand_history_with_channel.csv"
SKU_SOURCE    = "/content/sku_master.csv"        # or an s3:// URL
OUTPUT_FILE   = "GrowSari_Forecast_v4_Output.xlsx"

try:
    raw_demand = eng.load_any_table(DEMAND_SOURCE, "demand history", AWS_ACCESS_KEY_ID, AWS_SECRET_ACCESS_KEY, AWS_REGION)
    raw_skum   = eng.load_any_table(SKU_SOURCE, "SKU master", AWS_ACCESS_KEY_ID, AWS_SECRET_ACCESS_KEY, AWS_REGION)

    demand = eng.validate_and_normalize_demand(raw_demand)
    skum   = eng.validate_and_normalize_sku_master(raw_skum)
    demand = eng.merge_demand_with_sku(demand, skum)

    REF_DATE = demand["date"].max()
    HOL = eng.build_ph_holidays(demand["date"].min().year, REF_DATE.year + 1)
    REG = {d for d, v in HOL.items() if v[1] == "regular"}
    SPC = {d for d, v in HOL.items() if v[1] == "special"}

    CLOSED_DOWS = eng.detect_closed_dows(demand)
    TRADING_DOW = tuple(d for d in range(7) if d not in CLOSED_DOWS)
    eng.DEFAULT_PARAMS["closed_dows"] = CLOSED_DOWS
    _DOWN = ["Mon","Tue","Wed","Thu","Fri","Sat","Sun"]
    print("Trading weekdays:", [_DOWN[d] for d in TRADING_DOW],
          "| closed:", ([_DOWN[d] for d in CLOSED_DOWS] or "none (open 7 days)"))

    CLASSIF = eng.classify_all(demand, REF_DATE)
    print(f"rows: {len(demand)} | range: {demand.date.min().date()} -> {REF_DATE.date()}")
    for lv, df in CLASSIF.items():
        print(f"   {lv:22s} {len(df):4d} series")
except ValidationError as e:
    print("VALIDATION ERROR:", e)
    raise


## 3 · Optional: mount Drive (only if your source files live there)

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    print("Not running in Colab, or Drive not needed — skipping.")


## 4 · Input panel — choose your run settings

In [ ]:
_hist_months = sorted(demand["date"].dt.to_period("M").astype(str).unique())
_future_months = [(REF_DATE.to_period("M") + i).strftime("%Y-%m") for i in range(0, 13)]
_all_months = sorted(set(_hist_months) | set(_future_months))
_last_full = (REF_DATE.replace(day=1) - pd.Timedelta(days=1)).strftime("%Y-%m")
_next_month = (REF_DATE.to_period("M") + 1).strftime("%Y-%m")
GRAINS = ["daily","weekly","monthly"]; _ord = {g:i for i,g in enumerate(GRAINS)}

PANEL = {}
if HAS_WIDGETS:
    lay = W.Layout(width="460px"); st = {"description_width":"190px"}
    PANEL["level"]    = W.Dropdown(options=list(eng.LEVEL_KEYS), value="WH_SKU",
                                   description="1) Aggregation level", layout=lay, style=st)
    PANEL["temporal"] = W.Dropdown(options=GRAINS, value="daily",
                                   description="3) Temporal aggregation", layout=lay, style=st)
    PANEL["fgrain"]   = W.Dropdown(options=GRAINS, value="weekly",
                                   description="5a) Forecast granularity (weekly = custom 5-day week)", layout=lay, style=st)
    PANEL["testmonths"] = W.SelectMultiple(options=_hist_months, value=(_last_full,),
                                   rows=6, description="4) Test month(s)", layout=lay, style=st)
    PANEL["fc_start"] = W.Dropdown(options=_all_months, value=_next_month,
                                   description="5b) Forecast start", layout=lay, style=st)
    PANEL["fc_end"]   = W.Dropdown(options=_all_months, value=_next_month,
                                   description="5c) Forecast end", layout=lay, style=st)
    PANEL["upload"]   = W.FileUpload(accept=".xlsx", multiple=False,
                                   description="2) Combinations .xlsx")
    PANEL["metric"]   = W.Dropdown(options=["weighted_accuracy","minmax_agg","mape","smape","mae","rmse"],
                                   value="weighted_accuracy", description="Select-by metric", layout=lay, style=st)
    PANEL["transform"]= W.Dropdown(options=["log1p","none"], value="log1p",
                                   description="Pre-transform", layout=lay, style=st)
    def _sync(*_):
        t = PANEL["temporal"].value
        PANEL["fgrain"].options = [g for g in GRAINS if _ord[g] >= _ord[t]]
        if _ord[PANEL["fgrain"].value] < _ord[t]: PANEL["fgrain"].value = t
    PANEL["temporal"].observe(_sync, "value"); _sync()
    display(W.VBox([PANEL["level"], PANEL["temporal"], PANEL["fgrain"],
                    PANEL["testmonths"], PANEL["fc_start"], PANEL["fc_end"],
                    PANEL["metric"], PANEL["transform"], PANEL["upload"]]))
else:
    print("Widgets unavailable — edit FALLBACK_* in the BUILD CONFIG cell instead.")


## 5 · Combinations template (run after choosing the level)

In [ ]:
_lvl = PANEL["level"].value if HAS_WIDGETS else "WH_SKU"
eng.validate_level_keys(_lvl, eng.LEVEL_KEYS, demand)
_keys = eng.LEVEL_KEYS[_lvl]
_cols = _keys + [c for c in ["segment","ABC","lifecycle","total_units"] if c in CLASSIF[_lvl].columns]
_tmpl = CLASSIF[_lvl].sort_values("total_units", ascending=False)[_cols].reset_index(drop=True)
_tmpl_file = f"combinations_template_{_lvl}.xlsx"
_tmpl.to_excel(_tmpl_file, index=False)
print(f"Template for level '{_lvl}' -> {_tmpl_file}  ({len(_tmpl)} valid combinations)")
print("Required columns:", _keys)
try:
    from google.colab import files; files.download(_tmpl_file)
except Exception:
    pass
_tmpl.head(8)


## 6 · Build config from your inputs (validates the uploaded combinations)

In [ ]:
FALLBACK_LEVEL="WH_SKU"; FALLBACK_TEMPORAL="daily"; FALLBACK_FGRAIN="weekly"
FALLBACK_TESTMONTHS=[_last_full]; FALLBACK_FCSTART=_next_month; FALLBACK_FCEND=_next_month
FALLBACK_METRIC="weighted_accuracy"; FALLBACK_TRANSFORM="log1p"
FALLBACK_COMBINATIONS_PATH=None        # e.g. "my_combinations.xlsx" or an s3:// URL
FALLBACK_TOPN_IF_EMPTY=10

def _v(key, fb):  return PANEL[key].value if (HAS_WIDGETS and key in PANEL) else fb

LEVEL     = _v("level", FALLBACK_LEVEL)
TEMPORAL  = _v("temporal", FALLBACK_TEMPORAL)
FGRAIN    = _v("fgrain", FALLBACK_FGRAIN)
TESTMON   = list(_v("testmonths", FALLBACK_TESTMONTHS)) or [FALLBACK_TESTMONTHS[0]]
FC_S      = _v("fc_start", FALLBACK_FCSTART)
FC_E      = _v("fc_end", FALLBACK_FCEND)
METRIC    = _v("metric", FALLBACK_METRIC)
TRANSFORM = _v("transform", FALLBACK_TRANSFORM)
KEYS      = eng.LEVEL_KEYS[LEVEL]

try:
    eng.validate_level_keys(LEVEL, eng.LEVEL_KEYS, demand)
    eng.validate_grains(TEMPORAL, FGRAIN)
    eng.validate_date_window("test", TESTMON, _hist_months)
except ValidationError as e:
    print("VALIDATION ERROR:", e); raise

def _read_upload():
    up = PANEL.get("upload") if HAS_WIDGETS else None
    if up and len(up.value):
        v = up.value
        item = (list(v.values())[0] if isinstance(v, dict) else v[0])
        return pd.read_excel(io.BytesIO(item["content"]))
    if FALLBACK_COMBINATIONS_PATH:
        return eng.load_any_table(FALLBACK_COMBINATIONS_PATH, "combinations",
                                   AWS_ACCESS_KEY_ID, AWS_SECRET_ACCESS_KEY, AWS_REGION)
    return None

_raw = _read_upload()
try:
    if _raw is not None:
        COMBOS = eng.validate_combinations_upload(_raw, LEVEL, KEYS, demand, CLASSIF[LEVEL])
        not_found = COMBOS[~COMBOS["_found"]] if "_found" in COMBOS else COMBOS.iloc[0:0]
        if len(not_found): print("WARNING: not in data (skipped):\n", not_found[KEYS].to_string(index=False))
        COMBOS = COMBOS[COMBOS["_found"]].reset_index(drop=True)
        print(f"Loaded {len(COMBOS)} uploaded combinations for {LEVEL}.")
    else:
        COMBOS = eng.classify_all.__wrapped__ if False else None  # placeholder, overwritten below
        cols = KEYS + [c for c in ["segment","ABC","lifecycle","total_units"] if c in CLASSIF[LEVEL].columns]
        COMBOS = CLASSIF[LEVEL].sort_values("total_units", ascending=False)[cols].head(FALLBACK_TOPN_IF_EMPTY).reset_index(drop=True)
        print(f"No upload found -> using top {len(COMBOS)} combinations by volume (demo fallback).")
except ValidationError as e:
    print("VALIDATION ERROR:", e); raise

TESTMON = sorted(TESTMON)
TEST_START = pd.Timestamp(TESTMON[0] + "-01")
TEST_END   = (pd.Timestamp(TESTMON[-1] + "-01") + pd.offsets.MonthEnd(0))
FC_START   = pd.Timestamp(FC_S + "-01")
FC_END     = (pd.Timestamp(FC_E + "-01") + pd.offsets.MonthEnd(0))
if FC_END <= REF_DATE:
    print(f"NOTE: forecast window ends {FC_END.date()} <= data end {REF_DATE.date()} — forecast may be empty.")
if FC_END < FC_START:
    raise ValidationError("Forecast end month must be on/after the forecast start month.")

CFG = dict(TEMPORAL_AGG=TEMPORAL, FORECAST_AGG=FGRAIN,
           TEST_START=TEST_START, TEST_END=TEST_END, FC_START=FC_START, FC_END=FC_END,
           PRE_TRANSFORM=TRANSFORM, SELECTION_METRIC=METRIC,
           RUN_MODELS=["Naive","SeasonalNaive","MovingAvg","SeasonalRollAvg","ETS",
                       "Croston","SBA","TSB","XGBoost","LightGBM","Ensemble"])
print(f"\nLEVEL={LEVEL} | model grain={TEMPORAL} -> forecast grain={FGRAIN} (custom 5-day week if 'weekly')")
print(f"TEST  : {TEST_START.date()} .. {TEST_END.date()}  (months: {TESTMON})")
print(f"FCAST : {FC_START.date()} .. {FC_END.date()}  | select-by: {METRIC}")
print(f"Combinations to run: {len(COMBOS)}")


## 7 · RUN — all models on every combination (this is the heavy step)

In [ ]:
def series_rows(keyvals):
    mask = np.ones(len(demand), bool)
    for k, v in zip(KEYS, keyvals): mask &= (demand[k] == v)
    return demand[mask]

BT, LB, FCAST = [], [], []
t0 = time.time(); n = len(COMBOS)
for i, (_, r) in enumerate(COMBOS.iterrows(), 1):
    keyvals = [r[k] for k in KEYS]
    label = LEVEL + "|" + "|".join(str(v) for v in keyvals)
    meta = {"level": LEVEL, **{k: r[k] for k in KEYS},
            "segment": r.get("segment"), "ABC": r.get("ABC")}
    daily = eng.build_daily_series(series_rows(keyvals), TRADING_DOW)
    res = eng.run_series(daily, label, meta, CFG, HOL, REG, SPC, TRADING_DOW)
    if res:
        BT.append(res["backtest"]); LB.append(res["leaderboard"]); FCAST.append(res["forecast"])
    print(f"  [{i}/{n}] {label}", end="\r")
print(f"\nDone in {time.time()-t0:.0f}s")

BACKTEST  = pd.concat(BT, ignore_index=True)    if BT    else pd.DataFrame()
LEADER    = pd.concat(LB, ignore_index=True)    if LB    else pd.DataFrame()
FORECASTS = pd.concat([f for f in FCAST if len(f)], ignore_index=True) if any(len(f) for f in FCAST) else pd.DataFrame()
print("leaderboard rows:", len(LEADER), "| forecast rows:", len(FORECASTS))


## 8 · Champion selection & accuracy summary

In [ ]:
CHAMPIONS = LEADER[LEADER.is_champion].copy().sort_values("test_volume", ascending=False) if not LEADER.empty else pd.DataFrame()
FORECAST_CHAMPION = FORECASTS[FORECASTS.is_champion].copy() if not FORECASTS.empty else pd.DataFrame()

if not CHAMPIONS.empty:
    print("=== Champion model mix ===")
    print(CHAMPIONS["model"].value_counts().to_string())
    print("\n=== Champion accuracy (per series) ===")
    show = CHAMPIONS[["series","segment","ABC","model","weighted_accuracy","minmax_agg","mape","mae"]]
    print(show.round(3).to_string(index=False))
    print(f"\nVolume-weighted champion accuracy: "
          f"{np.average(CHAMPIONS['weighted_accuracy'].fillna(0), weights=CHAMPIONS['test_volume']):.3f}")
if not FORECAST_CHAMPION.empty:
    print("\n=== Forecast (champion) — selected window @", CFG["FORECAST_AGG"], "===")
    print(FORECAST_CHAMPION[["series","model","period","forecast"]]
          .assign(period=lambda d: d.period.dt.date).round(1).to_string(index=False))


## 9 · Plots

In [ ]:
if not BACKTEST.empty and not CHAMPIONS.empty:
    ex = CHAMPIONS["series"].iloc[0]; champ = CHAMPIONS["model"].iloc[0]
    sub = BACKTEST[(BACKTEST.series==ex)&(BACKTEST.model==champ)].sort_values("period")
    fig, ax = plt.subplots(figsize=(11,4))
    ax.plot(sub.period, sub.actual, "k-o", label="Actual (test)")
    ax.plot(sub.period, sub.pred, "r--s", label=f"{champ} (champion)")
    fcw = FORECAST_CHAMPION[FORECAST_CHAMPION.series==ex].sort_values("period")
    if not fcw.empty: ax.plot(fcw.period, fcw.forecast, "g-^", label="Forecast (future)")
    ax.set_title(f"{ex}  [{CFG['TEMPORAL_AGG']}->{CFG['FORECAST_AGG']}]"); ax.legend(); ax.grid(alpha=.3)
    plt.tight_layout(); plt.show()


## 10 · Export — Excel, including the custom-week output sheet

`Output_CustomWeek` matches the requested shape: depot · sku · model · data_type (test/forecast) · date · month (e.g. 'Mar') · year, with dates following the 5-day / 6-per-month custom calendar.

In [ ]:
run_cfg = pd.DataFrame({"parameter": ["level","temporal_agg","forecast_agg","test_window",
        "forecast_window","select_metric","pre_transform","n_combinations","data_range"],
    "value": [LEVEL, TEMPORAL, FGRAIN, f"{TEST_START.date()}..{TEST_END.date()}",
              f"{FC_START.date()}..{FC_END.date()}", METRIC, TRANSFORM, len(COMBOS),
              f"{demand.date.min().date()}..{REF_DATE.date()}"]})

export_rows = []
if not FORECAST_CHAMPION.empty:
    fc_out = eng.attach_output_calendar_columns(FORECAST_CHAMPION, "period")
    fc_out["data_type"] = "forecast"
    fc_out = fc_out.rename(columns={"warehouse_id": "depot"}) if "warehouse_id" in fc_out.columns else fc_out
    export_rows.append(fc_out)
if not BACKTEST.empty and not CHAMPIONS.empty:
    champ_map = CHAMPIONS.set_index("series")["model"].to_dict()
    bt_champ = BACKTEST[BACKTEST.apply(lambda r: champ_map.get(r["series"]) == r["model"], axis=1)]
    bt_out = eng.attach_output_calendar_columns(bt_champ, "period")
    bt_out["data_type"] = "test"
    export_rows.append(bt_out)
OUTPUT_CUSTOM_WEEK = pd.concat(export_rows, ignore_index=True) if export_rows else pd.DataFrame()

with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl") as xl:
    run_cfg.to_excel(xl, sheet_name="Run_Config", index=False)
    if not CHAMPIONS.empty: CHAMPIONS.round(4).to_excel(xl, sheet_name="Champions", index=False)
    if not LEADER.empty:    LEADER.round(4).to_excel(xl, sheet_name="Leaderboard", index=False)
    if not FORECAST_CHAMPION.empty:
        FORECAST_CHAMPION.round(2).to_excel(xl, sheet_name="Forecast_Champion", index=False)
    if not FORECASTS.empty: FORECASTS.round(2).to_excel(xl, sheet_name="Forecast_AllModels", index=False)
    if not BACKTEST.empty:  BACKTEST.round(2).to_excel(xl, sheet_name="Backtest_Detail", index=False)
    if not OUTPUT_CUSTOM_WEEK.empty:
        OUTPUT_CUSTOM_WEEK.round(2).to_excel(xl, sheet_name="Output_CustomWeek", index=False)
    CLASSIF[LEVEL].round(4).to_excel(xl, sheet_name=("CL_"+LEVEL)[:31], index=False)
print("written ->", OUTPUT_FILE)
try:
    from google.colab import files; files.download(OUTPUT_FILE)
except Exception:
    pass


---
**Recap of the input contract**
- *Level* sets the required columns for the upload: `WH_SKU`→(warehouse_id, sku_id); `WH_SKU_Channel`→(+channel); `WH_Category`→(warehouse_id, category); `WH_Category_Channel`→(+channel). Header variants (e.g. `Warehouse`, `SKU`, `Depot`) are auto-mapped, and any mismatch is caught with an explicit error before the engine runs.
- *Demand history / SKU master / combinations* can each be a local upload **or** an `s3://bucket/key` URL — large files are streamed in chunks rather than loaded fully into memory.
- *Test month(s)* must exist in history (they are scored); unknown months raise a validation error immediately. *Forecast window* may be future; the forecast is sliced to those month(s) at the chosen granularity.
- *Forecast granularity* is constrained to be ≥ the temporal aggregation. When set to `weekly`, buckets follow the **custom 5-day / 6-buckets-per-month calendar** (days 1-5, 6-10, 11-15, 16-20, 21-25, 26-end), labelled by the bucket's last day — e.g. `2026-03-05 ... 2026-03-31, 2026-04-05 ...`.
